# Roteiro: Engenheiro de Dados
## FASE 5 — Apache Airflow

**O que é Airflow:** Orquestrador de pipelines. Define **quando**, **em que ordem** e **o que fazer** se uma tarefa falhar.

**Relação com o dbt:** O dbt transforma os dados. O Airflow agenda e monitora quando o dbt roda.

```
Airflow (agenda e monitora)
    └── dbt run staging
    └── dbt run marts  
    └── dbt test
```

---
**Estrutura criada:**
```
airflow/
├── docker-compose.yml          ← sobe o Airflow via Docker
├── dags/
│   └── pipeline_contoso.py    ← seu pipeline orquestrado
├── logs/                       ← logs gerados automaticamente
└── plugins/                    ← extensões (vazio por ora)
```

---
**Índice:**
1. Conceitos fundamentais
2. Pré-requisitos
3. Subir o Airflow com Docker
4. Acessar a interface web
5. Entender o DAG criado
6. Rodar o pipeline manualmente
7. Entender os logs
8. Comandos essenciais

---
## 1. Conceitos Fundamentais

### DAG — Directed Acyclic Graph
É o pipeline. Define as tarefas e a ordem de execução.

```
inicio → verifica_banco → dbt_staging → dbt_marts → dbt_test → fim
```

**Directed** = tem direção (seta de A para B)  
**Acyclic** = não volta (não cria loop)  
**Graph** = conjunto de tarefas conectadas

### Task — Tarefa
Cada caixa dentro do DAG é uma tarefa. Pode ser:

| Operator | O que faz |
|---|---|
| `PythonOperator` | Roda uma função Python |
| `BashOperator` | Roda um comando de terminal |
| `SQLExecuteQueryOperator` | Roda uma query SQL |
| `EmailOperator` | Envia e-mail |

### Schedule
Define quando o DAG roda. Usa sintaxe cron:

```python
schedule_interval='0 6 * * *'   # todo dia às 6h
schedule_interval='0 8 * * 1'   # toda segunda às 8h
schedule_interval='@daily'      # todo dia
schedule_interval='@hourly'     # toda hora
schedule_interval=None          # só manual
```

---
## 2. Pré-requisitos

Airflow não roda nativamente no Windows — precisa do **Docker**.

**Verificar se Docker está instalado:**

In [ ]:
import subprocess

result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print('Docker instalado:', result.stdout.strip())
else:
    print('Docker NÃO encontrado.')
    print('Instale em: https://www.docker.com/products/docker-desktop')

result2 = subprocess.run(['docker-compose', '--version'], capture_output=True, text=True)
if result2.returncode == 0:
    print('Docker Compose instalado:', result2.stdout.strip())
else:
    result3 = subprocess.run(['docker', 'compose', 'version'], capture_output=True, text=True)
    if result3.returncode == 0:
        print('Docker Compose instalado:', result3.stdout.strip())

---
## 3. Subir o Airflow com Docker

Rode os comandos abaixo **no terminal** dentro da pasta `airflow/`.

```bash
cd "c:\Users\PDCASE\Desktop\SQL VSCode\airflow"

# Primeira vez — inicializa o banco do Airflow
docker compose up airflow-init

# Sobe todos os serviços
docker compose up -d
```

**O que sobe:**
- `postgres` — banco de dados interno do Airflow
- `airflow-webserver` — interface web na porta 8080
- `airflow-scheduler` — o agendador que roda os DAGs

**Para parar:**
```bash
docker compose down
```

In [ ]:
# Verificar se os containers estão rodando
result = subprocess.run(
    ['docker', 'compose', 'ps'],
    capture_output=True, text=True,
    cwd=r'c:\Users\PDCASE\Desktop\SQL VSCode\airflow'
)
print(result.stdout)

---
## 4. Acessar a Interface Web

Depois que os containers subirem:

```
URL:   http://localhost:8080
Login: admin
Senha: admin
```

Na interface você vai ver o DAG `pipeline_contoso` listado.

In [ ]:
# Abrir a interface do Airflow no navegador
import webbrowser
webbrowser.open('http://localhost:8080')
print('Abrindo http://localhost:8080')
print('Login: admin | Senha: admin')

---
## 5. Entender o DAG criado

Arquivo: `airflow/dags/pipeline_contoso.py`

### Ordem de execução

```
notificar_inicio
       ↓
verificar_banco        ← testa conexão com SQL Server
       ↓
dbt_run_staging        ← dbt run --select staging
       ↓
dbt_run_marts          ← dbt run --select marts
       ↓
dbt_test               ← dbt test
       ↓
notificar_fim
```

### Como a ordem é definida no código

```python
t1_inicio >> t2_verificar_banco >> t3_dbt_staging >> t4_dbt_marts >> t5_dbt_test >> t6_fim
```

O operador `>>` significa **"roda antes de"**. Se qualquer tarefa falhar, as próximas não rodam.

---
## 6. Rodar o Pipeline Manualmente

Na interface web:
1. Clique no DAG `pipeline_contoso`
2. Clique no botão **▶ Trigger DAG** (canto superior direito)
3. Acompanhe cada tarefa ficando verde

Ou pelo terminal:
```bash
docker compose exec airflow-webserver airflow dags trigger pipeline_contoso
```

In [ ]:
# Disparar o DAG pelo Python
result = subprocess.run(
    ['docker', 'compose', 'exec', 'airflow-webserver',
     'airflow', 'dags', 'trigger', 'pipeline_contoso'],
    capture_output=True, text=True,
    cwd=r'c:\Users\PDCASE\Desktop\SQL VSCode\airflow'
)
print(result.stdout)
if result.returncode != 0:
    print('ERRO:', result.stderr)

---
## 7. Entender os Logs

Cada tarefa gera um log. Na interface web:
1. Clique no DAG
2. Clique em uma execução
3. Clique em uma tarefa
4. Clique em **Log**

Os logs também ficam salvos em `airflow/logs/`.

### Estados de uma tarefa

| Cor | Estado | Significado |
|---|---|---|
| Cinza | `queued` | Aguardando execução |
| Amarelo | `running` | Executando agora |
| Verde | `success` | Concluiu com sucesso |
| Vermelho | `failed` | Falhou |
| Laranja | `up_for_retry` | Vai tentar de novo |

---
## 8. Comandos Essenciais

Todos rodados dentro da pasta `airflow/`:

```bash
# Subir o Airflow
docker compose up -d

# Parar o Airflow
docker compose down

# Ver containers rodando
docker compose ps

# Ver logs do scheduler
docker compose logs airflow-scheduler

# Listar todos os DAGs
docker compose exec airflow-webserver airflow dags list

# Disparar um DAG manualmente
docker compose exec airflow-webserver airflow dags trigger pipeline_contoso

# Ver histórico de execuções
docker compose exec airflow-webserver airflow dags list-runs -d pipeline_contoso
```

---
## O que o Airflow resolve na prática

**Sem Airflow:**
```
- Script agendado no Task Scheduler
- Ninguém sabe se rodou ou falhou
- Se falhou, ninguém sabe em qual etapa
- Sem retry automático
- Sem histórico
```

**Com Airflow:**
```
- Interface visual mostrando cada execução
- Log detalhado de cada tarefa
- Retry automático se falhar
- Histórico completo de execuções
- Alertas de falha por e-mail
- Dependência entre tarefas garantida
```

---
## Critério para avançar para a Fase 6 (Portfólio)

- [ ] Docker instalado e funcionando
- [ ] Airflow subindo com `docker compose up`
- [ ] DAG `pipeline_contoso` aparecendo na interface
- [ ] Conseguir disparar o DAG manualmente e ver as tarefas
- [ ] Entender o que cada tarefa do DAG faz